# Datamine WIREPE Process Example

This notebook demonstrates how to configure and run the **`wirepe`** process wrapper in `dmstudio`.

## Process Description

## WIREPE

"Section" a wireframe to produce a set of perimeters (closed strings) at various section positions.

### Input Files:

* **wiretr** (Wireframe Triangle):
Input wireframe triangle file.
Required=Yes

* **wirept** (Wireframe Points):
Input wireframe point file.
Required=Yes

* **section** (Section):
Optional section definition file. Required fields: SVALUE - used to select specific
section. XCENTRE,YCENTRE - coords of centre of section. SDIP - dip of section (degrees).
SAZI - azimuth of dip direction (degrees). STHICK - influence of section (not used).
Required=No

### Output Files:

* **perimout** (String):
Output perimeter file. File contains XP, YP, ZP, PVALUE, PTN and fields copied from the
triangle file as specified by optional field specifications ATTRIB1, ATTRIB2, ATTRIB3
and ATTRIB4.
Required=Yes

### Fields:

* **attribs** (Undefined : Undefined):
Field #1 to be copied from triangle to perimeter file.
Default=Undefined
Required=No

### Parameters:

* **xincr**:
Increment along the X axis. Parameter is used, in conjunction with YINCR and ZINCR to
calculate an offset from the specified section to create a family of sections.
Range=Undefined
Values=Undefined
Default=Undefined
Required=No

* **yincr**:
Increment along the Y axis. See XINCR.
Range=Undefined
Values=Undefined
Default=Undefined
Required=No

* **zincr**:
Increment along the Z axis. See XINCR.
Range=Undefined
Values=Undefined
Default=Undefined
Required=No

* **pincr**:
An alternate form of specifiying an offset from the specified section to form a family
of parallel sections.
Range=Undefined
Values=Undefined
Default=Undefined
Required=No

* **azincr**:
Increment azimuth from 0 to 180 about the centre point of the section, including the
section defined by AZI.
Range=0,180
Values=Undefined
Default=Undefined
Required=No

* **dipincr**:
Increment dip from - 90 to + 90 about the centre point of the section, including the
section defined by DIP.
Range=-90, 90
Values=Undefined
Default=Undefined
Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('wirepe')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute wirepe
print("Running wirepe...")
dm_cmd.wirepe(
    wiretr_i='t_SurfaceTriangles',  # required
    wirept_i='t_SurfaceTriangles',  # required
    section_i='optional',  # required
    perimout_o='t_wirepe_out',  # required
    # attribs_f=['optional'],  # optional
    # xincr_p='optional',  # optional
    # yincr_p='optional',  # optional
    # zincr_p='optional',  # optional
    # pincr_p='optional',  # optional
    # azincr_p='optional',  # optional
    # dipincr_p='optional',  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("wirepe execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_wirepe_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")